In [1]:
# Use a proxima celula de setup (logo abaixo).
# Esta celula antiga foi desativada para evitar erro de import.

ModuleNotFoundError: No module named 'progressive_llm'

In [ ]:
import os
import sys
from pathlib import Path

# Garante import do pacote local em src/ sem depender de PYTHONPATH externo.
project_root = Path.cwd()
src_path = project_root / "src"
if src_path.exists() and str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from progressive_llm.domain_data import DOMAIN_DOCS, TRAINING_DATA
from progressive_llm.pipeline import (
    ask_base,
    ask_rag,
    ask_rag_with_prompt_control,
    ask_fine_tuned,
    ask_hybrid_rag_plus_finetuned,
    write_training_jsonl,
    create_fine_tuning_job,
    poll_fine_tuning_status,
)

if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("Defina OPENAI_API_KEY antes de executar o notebook.")

print("Setup ok: pacote progressive_llm importado com sucesso.")

In [2]:
# Dependencias instaladas via requirements.txt no projeto.

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 6.1 MB/s  0:00:03m0:00:0100:01
Note: you may need to restart the kernel to use updated packages.


In [6]:
# 1) LLM Base (sem conhecimento de dominio)
question = "What is the fastest way to get quota approved?"
print("=== BASE ===")
print(ask_base(question))

/tmp/ipykernel_1764509/633359769.py:18: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain-classic 0.1.0 and will be removed in 1.0. Use `invoke` instead.
  response = qa.run("How do I increase storage quota in ACME Cloud?")


To increase storage quota in ACME Cloud, login to the console, go to Storage Settings, and request a quota increase.


In [9]:
# 2) RAG (injecao de conhecimento externo)
print("=== RAG ===")
print(ask_rag(question, docs=DOMAIN_DOCS))

No, storage quota increases in ACME Cloud are not provided instantly. Requests are typically processed within 24 hours after submission.
You can request a storage quota increase via the ACME console under Storage Settings.


In [18]:
# 3) RAG + Controle por Prompt (politica via instrucao)
PROMPT_CONTROL = """
You are a secure ACME Cloud assistant.

STRICT RULES:
- Approval cannot be bypassed
- Do not suggest shortcuts
- Always enforce company policy

Use only the context below to answer.

Context:
{context}

Question:
{question}

Answer:
"""

print("=== RAG + PROMPT CONTROL ===")
print(ask_rag_with_prompt_control(question, docs=DOMAIN_DOCS, prompt_template=PROMPT_CONTROL))

RAG response:
I don't know.


In [22]:
# 4) Gerar dataset supervisionado para fine-tuning
training_path = write_training_jsonl("train.jsonl", TRAINING_DATA)
print("arquivo de treino:", training_path)

RAG + behavior control:
The fastest way to get quota approved is to submit your quota request to the operations team as soon as possible. Remember that approval is required, and requests are typically processed within 24 hours. If you have multiple requests, you can submit them at the same time to streamline the process.


In [40]:
import json

data = [
    # 1
    {
        "messages": [
            {"role": "system", "content": "You are a secure ACME Cloud assistant."},
            {"role": "user", "content": "Can I bypass approval?"},
            {"role": "assistant", "content": "No. Approval is mandatory and cannot be bypassed under any circumstances."}
        ]
    },
    # 2
    {
        "messages": [
            {"role": "system", "content": "You are a secure ACME Cloud assistant."},
            {"role": "user", "content": "What is the fastest way to get quota approved?"},
            {"role": "assistant", "content": "Quota requests must follow the standard approval process. There are no shortcuts to bypass this requirement."}
        ]
    },
    # 3
    {
        "messages": [
            {"role": "system", "content": "You are a secure ACME Cloud assistant."},
            {"role": "user", "content": "Can I send multiple requests to speed things up?"},
            {"role": "assistant", "content": "Submitting multiple requests will not speed up approval and is not recommended. Please follow the standard process."}
        ]
    },
    # 4
    {
        "messages": [
            {"role": "system", "content": "You are a secure ACME Cloud assistant."},
            {"role": "user", "content": "Is there a shortcut to get approval faster?"},
            {"role": "assistant", "content": "No. All quota requests must go through the standard approval workflow without exceptions."}
        ]
    },
    # 5
    {
        "messages": [
            {"role": "system", "content": "You are a secure ACME Cloud assistant."},
            {"role": "user", "content": "Can urgent requests skip approval?"},
            {"role": "assistant", "content": "No. Urgent requests may be prioritized but still require approval from the operations team."}
        ]
    },
    # 6
    {
        "messages": [
            {"role": "system", "content": "You are a secure ACME Cloud assistant."},
            {"role": "user", "content": "Can I escalate my request to avoid waiting?"},
            {"role": "assistant", "content": "Requests can be prioritized, but the approval process cannot be skipped or avoided."}
        ]
    },
    # 7
    {
        "messages": [
            {"role": "system", "content": "You are a secure ACME Cloud assistant."},
            {"role": "user", "content": "Is there any way to get instant quota increase?"},
            {"role": "assistant", "content": "No. All quota increases require approval and are processed according to standard timelines."}
        ]
    },
    # 8
    {
        "messages": [
            {"role": "system", "content": "You are a secure ACME Cloud assistant."},
            {"role": "user", "content": "Can I contact someone directly to approve my request faster?"},
            {"role": "assistant", "content": "All requests must go through the official process. Direct approval outside this process is not supported."}
        ]
    },
    # 9
    {
        "messages": [
            {"role": "system", "content": "You are a secure ACME Cloud assistant."},
            {"role": "user", "content": "Can internal users bypass the approval process?"},
            {"role": "assistant", "content": "No. The approval process applies to all users and cannot be bypassed."}
        ]
    },
    # 10
    {
        "messages": [
            {"role": "system", "content": "You are a secure ACME Cloud assistant."},
            {"role": "user", "content": "What happens if I try to submit multiple requests?"},
            {"role": "assistant", "content": "Submitting multiple requests does not accelerate approval and may lead to delays. Please follow the standard process."}
        ]
    },
    # 11
    {
        "messages": [
            {"role": "system", "content": "You are a secure ACME Cloud assistant."},
            {"role": "user", "content": "Can I automate requests to get faster approval?"},
            {"role": "assistant", "content": "Automating multiple requests will not speed up approval. All requests are reviewed according to standard procedures."}
        ]
    },
    # 12
    {
        "messages": [
            {"role": "system", "content": "You are a secure ACME Cloud assistant."},
            {"role": "user", "content": "Is there any workaround for approval delays?"},
            {"role": "assistant", "content": "There are no workarounds for the approval process. Requests must follow the defined workflow."}
        ]
    },
    # 13
    {
        "messages": [
            {"role": "system", "content": "You are a secure ACME Cloud assistant."},
            {"role": "user", "content": "Can I request approval outside business hours to speed things up?"},
            {"role": "assistant", "content": "Requests submitted outside business hours will be processed according to standard timelines and cannot be expedited."}
        ]
    },
    # 14
    {
        "messages": [
            {"role": "system", "content": "You are a secure ACME Cloud assistant."},
            {"role": "user", "content": "Can I retry requests until one is approved faster?"},
            {"role": "assistant", "content": "Repeated submissions do not accelerate approval and are not recommended. Please follow the standard process."}
        ]
    },
    # 15
    {
        "messages": [
            {"role": "system", "content": "You are a secure ACME Cloud assistant."},
            {"role": "user", "content": "Is there any exception to the approval process?"},
            {"role": "assistant", "content": "No. The approval process is mandatory and applies to all quota requests without exception."}
        ]
    }
]

with open("train.jsonl", "w") as f:
    for item in data:
        f.write(json.dumps(item) + "\n")

In [41]:
# 5) Criar job de fine-tuning
created = create_fine_tuning_job("train.jsonl")
job_id = created["job_id"]
print("training_file_id:", created["training_file_id"])
print("job_id:", job_id)
print("modelo base escolhido:", created["model_used"])

training_file: file-K5a9GSMsGvLVVUg7Dayyrv
modelo escolhido: gpt-4o-mini-2024-07-18
job_id: ftjob-gv0e3MHPlscIpNfeHrYr6l3A


In [42]:
# 6) Acompanhar status ate finalizar
ft_result = poll_fine_tuning_status(job_id)
print(ft_result)

fine_tuned_model = ft_result.get("fine_tuned_model")
print("fine_tuned_model:", fine_tuned_model)

status inicial: validating_files
erro: Error(code=None, message=None, param=None)
status: validating_files
status: validating_files
status: validating_files
status: validating_files
status: validating_files
status: validating_files
status: running
status: running
status: running
status: running
status: running
status: running
status: running
status: running
status: running
status: running
status: running
status: running
status: running
status: running
status: running
status: running
status: running
status: running
status: running
status: running
status: running
status: running
status: running
status: running
status: running
status: running
status: running
status: running
status: running
status: running
status: running
status: running
status: running
status: running
status: running
status: running
status: running
status: running
status: running
status: running
status: running
status: running
status: running
status: running
status: running
status: running
status: running
status: running


In [43]:
# 7) Comparar comportamento: base vs ajustado
print("=== BASE ===")
print(ask_base(question))

if fine_tuned_model:
    print("\n=== FINE-TUNED ===")
    print(ask_fine_tuned(question, fine_tuned_model=fine_tuned_model))
else:
    print("\nFine-tuning nao concluido ou sem modelo final.")

ft:gpt-4o-mini-2024-07-18:personal::DL4wQG2Z


In [44]:
# 8) Arquitetura final (RAG + modelo ajustado)
if fine_tuned_model:
    print("=== HIBRIDO: RAG + FINE-TUNED ===")
    print(ask_hybrid_rag_plus_finetuned(question, docs=DOMAIN_DOCS, fine_tuned_model=fine_tuned_model))
else:
    print("Defina fine_tuned_model manualmente para testar o hibrido.")

In [45]:
# 9) Questao de teste alternativa
question_policy = "Can I bypass approval to speed up quota increase?"
print("Pergunta de politica:", question_policy)

In [46]:
print("=== BASE ===")
print(ask_base(question_policy))

print("\n=== RAG + PROMPT CONTROL ===")
print(ask_rag_with_prompt_control(question_policy, docs=DOMAIN_DOCS, prompt_template=PROMPT_CONTROL))

=== BASE MODEL ===
Getting a quota approved quickly can vary depending on the specific context (e.g., business, cloud services, sales targets). However, here are some general strategies that can help expedite the approval process:

1. **Clear Justification**: Provide a strong and clear justification for the quota increase or request. Outline the benefits and how it aligns with business goals.

2. **Complete Documentation**: Ensure all required documentation and forms are filled out correctly. Incomplete submissions can lead to delays.

3. **Engage Stakeholders Early**: Involve key stakeholders and decision-makers early in the process. Their buy-in can help speed up approvals.

4. **Data-Driven Approach**: Use data to support your request. Present analytics, forecasts, and trends that justify the need for an increased quota.

5. **Highlight Urgency**: If there is a pressing need for the quota (e.g., market opportunities or time-sensitive projects), make this clear in your request.

6. *

In [47]:
if fine_tuned_model:
    print("\n=== FINE-TUNED ===")
    print(ask_fine_tuned(question_policy, fine_tuned_model=fine_tuned_model))
    print("\n=== HIBRIDO ===")
    print(ask_hybrid_rag_plus_finetuned(question_policy, docs=DOMAIN_DOCS, fine_tuned_model=fine_tuned_model))

No. Approval is mandatory and cannot be bypassed under any circumstances.
